In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_contact_details_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ContactDetailId	String"
        #     ",SourceSystem	String"
        #     ",PortfolioKey	String"
        #     ",PortfolioName	String"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId	String"
        #     ",ContactDetailKey	String"
        #     ",CDI	string"
        #     ",PartyKey	String"
        #     ",ContactDetailTypeCode	string"
        #     ",ContactDetailType	string"
        #     ",ContactDetailTypeGroup string"
        #     ",FullAddress	string"
        #     ",Country	string"
        #     ",County	string"
        #     ",City	string"
        #     ",PostalCode	string"
        #     ",Street	string"
        #     ",Street1 string"
        #     ",Street2 string"
        #     ",Street3 string"
        #     ",StreetNumber	string"
        #     ",PhoneNumber1	string"
        #     ",PhoneNumber2	string"
        #     ",PhoneNumber3	string"
        #     ",EmailAddress1	string"
        #     ",EmailAddress2	string"
        #     ",IsValidFlag	boolean"
        #     ",CommunicationFlag	string"
        #     ",IsPrimaryFlag	boolean"
        #     ",CreatedBy	string"
        #     ",CreatedDate	timestamp"
        #     ",LastUpdatedDate	date"
        #     ",OriginalTelephoneCode	string"
        #     ",OriginalTelephoneType	string"
        #     ",IsActiveFlag	string"
        #     ",DoNotCallFlag	boolean"
        #     ",IsDialerActivityFlag	int"
        #     ",DialerActivityTelephoneNumber	string"
        #     ",InternalConfirmationTypeCode	int"
        #     ",InternalConfirmationType	string"
        #     ",CollectionSourceTypeCode	int"
        #     ",CollectionSourceType	string"
        #     ",DoNotSendFlag boolean"
        #     ",AddContactFlag boolean"
        #     ",SerialNumber int"
        #     ",BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]
        curated_table_path = f"{curated_zone}.ContactDetails"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "profits",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "cust_id",
            "cdi",
            "unit_code",
            "E_MAIL",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_customer",
            customer_final_filters,
            customer_selected_fields,
        )

        # ==============================customer_address====================
        customer_address_extra_filters = None
        if customer_address_extra_filters is not None:
            customer_address_extra_filters.append(date_filter)
            customer_address_final_filters = " AND ".join(
                customer_address_extra_filters
            )
        else:
            customer_address_final_filters = date_filter
        customer_address_selected_fields = [
            "cust_id",
            "serial_num",
            "address_type",
            "country",
            "addresstypelit",
            "state_province",
            "city",
            "zip_code",
            "address",
            "telephone_1",
            "telephone_2",
            "telephone_3",
            "address_valid",
            "communication_addr",
            "businessdate",
        ]
        customer_address = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_customer_address",
            customer_address_final_filters,
            customer_address_selected_fields,
        )

        # ==============================contact_details_profits====================

        contact_details_profits_primary_key_fields = [
            customer_address.cust_id,
            customer_address.addresstypelit,
            customer_address.serial_num,
            lit("profits"),
        ]

        contact_details_profits = customer.join(
            customer_address,
            (customer_address["cust_id"] == customer["cust_id"]),
            "left",
        ).join(
            portfolio,
            (portfolio["unit_code_profits"] == customer["unit_code"]),
            "left",
        )

        contact_details_profits = contact_details_profits.withColumn(
            "ContactDetailId",
            sha2(concat_ws(";", *contact_details_profits_primary_key_fields), 256),
        ).withColumn(
            "ContactDetailKey",
            concat_ws(
                ";",
                customer_address.cust_id,
                customer_address.addresstypelit,
                customer_address.serial_num,
            ),
        )

        contact_details_profits_df = contact_details_profits.select(
            contact_details_profits.ContactDetailId,
            lit("profits").alias("SourceSystem"),
            customer.unit_code.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            contact_details_profits.ContactDetailKey.cast("string").alias(
                "ContactDetailKey"
            ),
            customer.cdi.alias("CDI"),
            customer_address.cust_id.cast("string").alias("PartyKey"),
            contact_details_profits.address_type.alias("ContactDetailTypeCode"),
            contact_details_profits.addresstypelit.alias("ContactDetailType"),
            lit(None).alias("ContactDetailTypeGroup"),
            lit(None).alias("FullAddress"),
            contact_details_profits.country.alias("Country"),
            contact_details_profits.state_province.alias("County"),
            contact_details_profits.city.alias("City"),
            contact_details_profits.zip_code.alias("PostalCode"),
            contact_details_profits.address.alias("Street"),
            contact_details_profits.address.alias("Street1"),
            lit(None).alias("Street2"),
            lit(None).alias("Street3"),
            lit(None).alias("StreetNumber"),
            contact_details_profits.telephone_1.alias("PhoneNumber1"),
            contact_details_profits.telephone_2.alias("PhoneNumber2"),
            contact_details_profits.telephone_3.alias("PhoneNumber3"),
            customer.E_MAIL.cast("string").alias("EmailAddress1"),
            lit(None).alias("EmailAddress2"),
            contact_details_profits.address_valid.alias("IsValidFlag"),
            contact_details_profits.communication_addr.alias("CommunicationFlag"),
            lit(None).alias("IsPrimaryFlag"),
            lit(None).alias("CreatedBy"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("LastUpdatedDate"),
            lit(None).alias("OriginalTelephoneCode"),
            lit(None).alias("OriginalTelephoneType"),
            lit(None).alias("IsActiveFlag"),
            lit(None).alias("DoNotCallFlag"),
            lit(None).alias("IsDialerActivityFlag"),
            lit(None).alias("DialerActivityTelephoneNumber"),
            lit(None).alias("InternalConfirmationTypeCode"),
            lit(None).alias("InternalConfirmationType"),
            lit(None).alias("CollectionSourceTypeCode"),
            lit(None).alias("CollectionSourceType"),
            lit(None).alias("DoNotSendFlag"),
            lit(None).alias("AddContactFlag"),
            customer_address.serial_num.alias("SerialNumber"),
            customer_address.businessdate.alias("BusinessDate"),
        )
        # ================== Append ==================

        contact_details_profits_df.write.format("delta").clusterBy(
            "BusinessDate"
        ).mode("append").saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_contact_details_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)